# ALL Met Painting Eyes

## The Met API

https://metmuseum.github.io/

https://github.com/metmuseum/openaccess

**Please limit request rate to 80 requests per second.**

In [ ]:
from google.colab import userdata
PAT_XYZ = userdata.get("PAT_XYZ")

In [ ]:
!pip install mediapipe
!pip install ultralytics
!wget https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task
!mkdir json && wget -P json https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/json/mp_masks_definitions.json
!wget https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/utils.py
!wget https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/utils_paintings.py
!git clone https://{PAT_XYZ}@github.com/acervos-digitais/met-faces-data.git data
!cd data && git config user.name "Thiago Hersan" && git config user.email "thiago.hersan+github@gmail.com"

In [ ]:
from utils import get_combined_jsons

from utils_paintings import PaintingsUtils

DATA_DIR = "./data"
JSON_DIR = f"{DATA_DIR}/json"
IMG_DIR = f"{DATA_DIR}/image"

JSON_OBJECTS_DIR = f"{JSON_DIR}/objects"

## Painting Objects

- $15\text{,}178$ on website
- $15\text{,}050$ (±100) available in API
- $14\text{,}233$ (±50) have images (according to `hasImages=true` API query param)
- $9\text{,}015$ ($63\%$) actually have images that can be downloaded
- $\sim4\text{,}500$ ($50\%$) have faces
- $\sim4\text{,}100$ ($90\%$) have extractable eye locations/shapes

In [ ]:
obj_ids = PaintingsUtils.get_object_ids()
len(obj_ids)

## Get Object Metadata

In [ ]:
mPU = PaintingsUtils(JSON_DIR, IMG_DIR)

for cnt,oid in enumerate(obj_ids):
  if cnt % 16 == 0:
    print(f"{cnt} / {len(obj_ids)}")

  mPU.get_obj_data(oid)

## Get Faces, Landmarks and Eyes

In [ ]:
objects_data = get_combined_jsons(JSON_OBJECTS_DIR)
len(objects_data)

In [ ]:
mPU = PaintingsUtils(JSON_DIR, IMG_DIR)
mPU.init_face_detector()
mPU.init_face_landmarker()

for cnt,obj_data in enumerate(objects_data):
  if cnt % 16 == 0:
    print(f"{cnt} / {len(objects_data)}")

  if mPU.is_done(obj_data):
    continue

  img = mPU.get_image(obj_data["primaryImage"])
  if img is None:
    continue

  face_data = mPU.get_face_data(obj_data, img)
  if face_data is None:
    continue

  landmark_data = mPU.get_landmark_data(face_data, img)
  if landmark_data is None:
    continue

  mPU.get_eye_images(landmark_data, img)

## Checks

In [ ]:
mPU = PaintingsUtils(JSON_DIR, IMG_DIR)
objs = get_combined_jsons(JSON_OBJECTS_DIR)

no_imgs_s = set(mPU.no_imgs)
ye_imgs_s = set([o["objectID"] for o in objs])

print(len(no_imgs_s), len(ye_imgs_s))

print(len(no_imgs_s.intersection(ye_imgs_s)))

## Export csv

In [ ]:
import pandas as pd

CSV_DIR = f"{DATA_DIR}/csv"

objs = get_combined_jsons(JSON_OBJECTS_DIR)

objs_df = pd.json_normalize(objs).drop(columns=["artistRole", "objectBeginDate", "objectEndDate", "tags"])

mhmw = objs_df[["measurements.Height", "measurements.Width"]]

objs_df = objs_df.drop(columns=[c for c in objs_df.columns if c.startswith("measurements")])

objs_df[["measurements.Height", "measurements.Width"]] = mhmw

In [ ]:
objs_df.to_csv(f"{CSV_DIR}/objects.csv", index=False)